# Пошаговый отладчик агента динамической типизации

Этот ноутбук выполняет **каждый узел** двух графов агента по отдельности и показывает его вход, выход и накопленное состояние. Он работает с локальным примером без gold-разметки, реальной LLM через существующий gateway и локальной закреплённой HHEM для NLI.

Важно: ниже намеренно используются служебные методы с префиксом `_`. Это не обычный пользовательский интерфейс пакета, а прозрачный режим отладки: он позволяет остановиться между узлами и проверить промежуточные данные вручную. Ячейки `schema_overview` и `nli_answer` делают реальные вызовы; при отсутствии настроенной среды ноутбук завершится ошибкой, а не переключится на заглушки.

## 1. Подготовка окружения

Перед запуском выберите ядро **Python (.venv-local-live, 3.12)**. Системный Python 3.13 не подходит для надёжной работы локальной HHEM. Запустите PowerShell из `dynamic_typing_agent`, активируйте `.venv-local-live`, загрузите `env.local.ps1` в текущий процесс с временным обходом политики, затем откройте Jupyter из этого же процесса — тогда ядро унаследует переменные окружения.

Эта ячейка находит корень самостоятельного пакета, проверяет только наличие обязательных переменных окружения (не показывает их значения) и создаёт агент из `config/live-gateway-hhem.yaml`. Кэш и диагностические файлы остаются во временном каталоге.

In [7]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from pathlib import Path

from IPython.display import JSON, Markdown, display

candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
PACKAGE_ROOT = next((path for path in candidates if (path / 'src' / 'hallugraph_dynamic_typing').is_dir()), None)
if PACKAGE_ROOT is None:
    raise RuntimeError('Откройте ноутбук из dynamic_typing_agent/notebooks или dynamic_typing_agent/.')

if str(PACKAGE_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / 'src'))

from hallugraph_dynamic_typing.agent import DynamicTypingAgent, graph_from_fixture
from hallugraph_dynamic_typing.models import AnswerInput, FrozenRegistry, SourceInput

required_environment = ('HALLU_GATEWAY_URL', 'HALLU_GATEWAY_API_KEY', 'HALLU_TYPING_MODEL', 'HALLU_HHEM_MODEL_PATH')
missing_environment = [name for name in required_environment if not os.environ.get(name, '').strip()]
if missing_environment:
    raise RuntimeError('Не найдены переменные: ' + ', '.join(missing_environment) + '. Выберите ядро Python (.venv-local-live, 3.12) и запустите Jupyter из PowerShell после загрузки env.local.ps1.')

temporary_workspace = tempfile.TemporaryDirectory(prefix='dynamic-typing-live-notebook-')
WORKSPACE = Path(temporary_workspace.name)
live_config = PACKAGE_ROOT / 'config' / 'live-gateway-hhem.yaml'
agent = DynamicTypingAgent.from_yaml(
    live_config,
    cache_root=WORKSPACE / 'cache',
    artifacts_root=WORKSPACE / 'runs',
)
if agent.backend.value != 'live' or agent.nli_backend != 'hhem':
    raise RuntimeError('Ожидался live/HHEM профиль; fake-режим для этого ноутбука запрещён.')

def show(title: str, value) -> None:
    display(Markdown(f'### {title}'))
    if hasattr(value, 'model_dump'):
        value = value.model_dump(mode='json')
    display(JSON(value, expanded=False))

print(f'Корень пакета: {PACKAGE_ROOT}')
print(f'Временная рабочая папка: {WORKSPACE}')
print(f'Режим модели: {agent.backend.value}; режим NLI: {agent.nli_backend}')
print('Проверка пройдена: fake-режим не используется. Значения секретов не выводятся.')

Корень пакета: C:\Users\Kolya\Desktop\SMILES\HaluVSGraph_Eval\hallu_smiles_dynamic_entity_typing\dynamic_typing_agent
Временная рабочая папка: C:\Users\Kolya\AppData\Local\Temp\dynamic-typing-live-notebook-jm8w4_mw
Режим модели: live; режим NLI: hhem
Проверка пройдена: fake-режим не используется. Значения секретов не выводятся.


## 2. Выбор входного примера без разметки

Берём первый пример из открытого набора. В ячейке показан только вход агента: контекст, запрос, ответ и подготовленные графы. Файл с человеческими ожиданиями здесь не читается.

In [8]:
fixture_path = PACKAGE_ROOT / 'examples' / 'dynamic_typing_20.no_gold.jsonl'
cases = [json.loads(line) for line in fixture_path.read_text(encoding='utf-8').splitlines() if line.strip()]
case = cases[0]
show('Выбранный пример', case)

source = SourceInput(
    source_id=case['source_id'],
    context_raw=case['context'],
    query_raw=case.get('query', ''),
    context_graph=graph_from_fixture(graph_id=f"{case['case_id']}:context", role='context', payload=case['graphs']['context']),
    query_graph=graph_from_fixture(graph_id=f"{case['case_id']}:query", role='query', payload=case['graphs']['query']),
)
source_state = {'source': source.model_dump(mode='json'), 'artifacts': []}
show('Начальное состояние графа источника', source_state)

### Выбранный пример

<IPython.core.display.JSON object>

### Начальное состояние графа источника

<IPython.core.display.JSON object>

## 3. Граф источника: `validate_source`

Проверяются структура входа, роли графов и отсутствие запрещённых полей с gold-разметкой.

In [ ]:
update = agent._validate_source(source_state)
show('Выход validate_source', update)
source_state.update(update)
show('Накопленное состояние', source_state)

## 4. Граф источника: `source_cache`

По входу и версии набора промптов вычисляется ключ кэша. Для свежего временного каталога ожидается `hit: false` и маршрут `fresh`. При повторном выполнении последующих ячеек без сброса может появиться `cached`.

In [ ]:
update = agent._source_cache(source_state)
show('Выход source_cache', update)
source_state.update(update)
route = agent._route_source_cache(source_state)
print(f'Выбранный маршрут: {route}')
if route != 'fresh':
    raise RuntimeError('Кэш уже заполнен. Перезапустите ячейку подготовки окружения для пошагового свежего прогона.')

## 5. Граф источника: `segment_source`

Контекст и запрос делятся на неизменяемые текстовые фрагменты. Именно их идентификаторы позже становятся доказательствами для типов и NLI.

In [ ]:
update = agent._segment_source(source_state)
show('Выход segment_source', update)
source_state.update(update)
show('Фрагменты-доказательства', source_state['spans'])

## 6. Граф источника: `schema_overview`

Этот узел отправляет реальной LLM контекст, запрос, фрагменты-доказательства и строгий контракт ответа. Ниже будут видны сформированные сообщения, ответ модели и локальная проверка JSON-схемы. Это первый сетевой вызов ноутбука.

In [ ]:
update = agent._schema_overview(source_state)
show('Выход schema_overview', update)
source_state.update(update)

## 7. Граф источника: `derive_registry`

На этом шаге локальные правила извлекают только явно заданные типы из графов источника и запроса. Здесь удобно вручную проверить определения, родителей, уровни доказательности и назначения типов сущностям.

In [ ]:
update = agent._derive_registry(source_state)
show('Выход derive_registry', update)
source_state.update(update)
show('Полученные типы', source_state['types'])
show('Назначения типов в источнике', source_state['assignments'])

## 8. Граф источника: `freeze_registry`

Реестр получает идентификатор и контрольную сумму, записывается в кэш и становится неизменяемым входом для графа ответа. После этой точки текст ответа не может добавить новый тип в реестр.

In [ ]:
update = agent._freeze_registry(source_state)
show('Выход freeze_registry', update)
source_state.update(update)
registry = FrozenRegistry.model_validate(source_state['registry'])
show('Замороженный реестр', registry)
print('Контрольная сумма:', registry.registry_sha256)

## 9. Подготовка входа графа ответа

Ответ передаётся вместе с уже замороженным реестром. Попробуйте изменить `registry_sha256`: следующая проверка должна завершиться ошибкой, а не продолжить работу с подменёнными данными.

In [ ]:
answer = AnswerInput(
    source_id=case['source_id'],
    response_id=case['case_id'],
    response_raw=case['response'],
    answer_graph=graph_from_fixture(graph_id=f"{case['case_id']}:answer", role='answer', payload=case['graphs']['answer']),
    registry=registry,
)
answer_state = {'answer': answer.model_dump(mode='json'), 'artifacts': []}
show('Начальное состояние графа ответа', answer_state)

## 10. Граф ответа: `validate_answer`

Проверяются роль графа ответа, соответствие `source_id` и контрольная сумма замороженного реестра.

In [ ]:
update = agent._validate_answer(answer_state)
show('Выход validate_answer', update)
answer_state.update(update)

## 11. Граф ответа: `annotate_answer`

Каждой сущности ответа назначается только тип из замороженного реестра либо `unknown`. В этой ячейке полезно сравнить `type_ids` с типами из ячейки 8.

In [ ]:
update = agent._annotate_answer(answer_state)
show('Выход annotate_answer', update)
answer_state.update(update)
show('Аннотации ответа', answer_state['annotations'])

## 12. Граф ответа: `nli_answer`

Для утверждений о ранее неизвестном типе формируется гипотеза и выполняется трёхзначная проверка: `entailed`, `contradicted` или `neutral`. В этом ноутбуке проверку выполняет реальная локальная закреплённая HHEM; при первом вызове загрузка модели может занять время.

In [ ]:
update = agent._nli_answer(answer_state)
show('Выход nli_answer', update)
answer_state.update(update)
show('Результаты NLI', answer_state['nli_results'])

## 13. Граф ответа: `emit_answer` и итоговый след

Последний узел фиксирует, что аннотации и NLI готовы к выдаче. Ниже можно последовательно раскрывать `artifacts`: это тот же наблюдаемый след, который сериализуется в артефакт запуска.

In [ ]:
update = agent._emit_answer(answer_state)
show('Выход emit_answer', update)
answer_state.update(update)

show('След узлов графа источника', source_state['artifacts'])
show('След узлов графа ответа', answer_state['artifacts'])

print('Готово: все узлы выполнены локально и по одному.')

## Что можно менять безопасно

- В ячейке 2 выберите другой `case` из `cases`.
- Измените графы или текст в памяти и повторите прогон с новой временной рабочей папкой.
- Для проверки кэш-ветви не перезапускайте ячейку подготовки, а повторите шаги 3–8: `source_cache` выберет `cached`, после чего можно вызвать `freeze_registry`.

Не загружайте в этот ноутбук gold-разметку и не подменяйте live-профиль на fake: цель файла — воспроизводимый разбор реального пути агента, а не запуск эксперимента по метрикам.